# Choice 数据地图自动生成

这份 Notebook 把“数据地图先行”变成可重复运行的代码。它会自动扫描当前项目、SQLite 数据库和 OpenBB Provider，并生成一份可交付的 Excel/JSON 数据地图。

输出包括：

- 数据集地图：数据集、Choice 接口、目标表、唯一键、更新节奏、OpenBB 映射和当前状态；
- 字段映射：Choice 原始字段到公司标准字段及 OpenBB 字段的转换规则；
- 库表字段：从 SQLite 自动读取真实表结构；
- 落库与采集证据：当前各数据源行数、日期范围和最近采集记录；
- 数据质量门槛：非空、去重、OHLC、单位、数据库完整性和 OpenBB 读取；
- OpenBB 覆盖：`choice` 与 `qianji` Provider 的注册和读取状态。

说明：本 Notebook **不会登录 Choice、不会再次消耗接口流量、不会读取或导出账号密码/令牌**。只有“日线已在 SQLite 中出现真实 Choice 数据”时，才会标记为“已真实验证”；其他接口保持“已实现未验证”或“规划中”。

## 1. 定位项目、数据库和输出目录

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display
from dotenv import load_dotenv

print("Python路径：", sys.executable)
print("Python版本：", sys.version.split()[0])
print("Notebook当前目录：", Path.cwd().resolve())

# Notebook放在项目notebooks目录时无需修改。
# 若单独存放，请填写项目根目录，例如：
# PROJECT_ROOT_OVERRIDE = r"D:\OneDrive\桌面\qianji_openbb_mini"
PROJECT_ROOT_OVERRIDE = ""

# 通常无需修改；只有想扫描另一份数据库时才填写绝对路径。
# DB_PATH_OVERRIDE = r"D:\data\qianji_market.db"
DB_PATH_OVERRIDE = ""

# 通常无需修改；留空时写入项目 validation_output 目录。
OUTPUT_DIR_OVERRIDE = ""


def find_project_root(start: Path) -> Path:
    if PROJECT_ROOT_OVERRIDE.strip():
        candidate = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
        if (candidate / "src" / "qianji_data_mini").exists():
            return candidate
        raise FileNotFoundError(f"指定的项目目录不正确：{candidate}")

    for candidate in (start, *start.parents):
        if (candidate / "src" / "qianji_data_mini").exists():
            return candidate
    raise FileNotFoundError(
        "没有找到qianji_openbb_mini项目。请把Notebook放进项目notebooks文件夹，"
        "或填写PROJECT_ROOT_OVERRIDE。"
    )


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
ENV_PATH = PROJECT_ROOT / ".env"
if ENV_PATH.exists():
    load_dotenv(ENV_PATH, override=True)

if DB_PATH_OVERRIDE.strip():
    DB_PATH = Path(DB_PATH_OVERRIDE).expanduser().resolve()
else:
    configured_db = Path(os.getenv("QIANJI_DB_PATH", "./data/qianji_market.db"))
    DB_PATH = configured_db if configured_db.is_absolute() else (PROJECT_ROOT / configured_db).resolve()

OUTPUT_DIR = (
    Path(OUTPUT_DIR_OVERRIDE).expanduser().resolve()
    if OUTPUT_DIR_OVERRIDE.strip()
    else (PROJECT_ROOT / "validation_output").resolve()
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 明确告诉qianji Provider本次扫描哪一份数据库；不涉及任何密码。
os.environ["QIANJI_DB_PATH"] = str(DB_PATH)

print("项目根目录：", PROJECT_ROOT)
print("配置文件存在：", ENV_PATH.exists())
print("SQLite数据库：", DB_PATH)
print("数据库存在：", DB_PATH.exists())
print("输出目录：", OUTPUT_DIR)

Python路径： d:\minicoda3\envs\dm311\python.exe
Python版本： 3.11.14
Notebook当前目录： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\notebooks
项目根目录： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini
配置文件存在： True
SQLite数据库： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\data\qianji_market.db
数据库存在： True
输出目录： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\validation_output


## 2. 自动扫描 SQLite 表结构、落库统计和采集记录

In [2]:
import sqlite3


TABLE_INFO_COLUMNS = [
    "table_name", "column_order", "column_name", "data_type",
    "not_null", "default_value", "primary_key", "business_description",
]
SOURCE_STATUS_COLUMNS = [
    "source", "rows", "symbols", "first_date", "last_date", "last_ingested_at",
]
INGESTION_COLUMNS = [
    "run_id", "source", "requested_symbols", "start_date", "end_date",
    "received_rows", "stored_rows", "failed_symbols", "started_at", "finished_at",
]

COLUMN_DESCRIPTIONS = {
    "source": "原始数据来源；Choice真实数据必须为choice",
    "symbol": "证券代码，含交易所后缀",
    "trade_date": "交易日期，ISO格式YYYY-MM-DD",
    "adjustment": "复权口径：unadjusted/qfq/hfq",
    "open": "开盘价",
    "high": "最高价",
    "low": "最低价",
    "close": "收盘价",
    "volume": "成交量，公司标准单位为股",
    "amount": "成交额，公司标准单位为CNY元",
    "previous_close": "前收盘价",
    "change_percent": "涨跌幅；SQLite层存百分点，例如1.25表示1.25%",
    "currency": "币种",
    "timezone": "市场时区",
    "volume_unit": "成交量单位",
    "amount_unit": "成交额单位",
    "ingested_at": "进入标准库的UTC时间",
    "raw_json": "供应商原始记录，用于追溯",
    "run_id": "采集任务流水号",
    "requested_symbols": "本次请求证券列表",
    "start_date": "请求开始日期",
    "end_date": "请求结束日期",
    "received_rows": "供应商返回记录数",
    "stored_rows": "本次写入或更新记录数",
    "failed_symbols": "失败证券及错误摘要，不含凭据",
    "started_at": "采集开始时间",
    "finished_at": "采集结束时间",
}


def table_exists(connection: sqlite3.Connection, table_name: str) -> bool:
    row = connection.execute(
        "SELECT 1 FROM sqlite_master WHERE type='table' AND name=?", (table_name,)
    ).fetchone()
    return row is not None


table_info_rows = []
source_status_df = pd.DataFrame(columns=SOURCE_STATUS_COLUMNS)
ingestion_df = pd.DataFrame(columns=INGESTION_COLUMNS)
choice_df = pd.DataFrame()
db_integrity = "数据库不存在"
database_tables = []

if DB_PATH.exists():
    try:
        with sqlite3.connect(DB_PATH) as connection:
            db_integrity = str(connection.execute("PRAGMA quick_check").fetchone()[0])
            database_tables = [
                row[0]
                for row in connection.execute(
                    "SELECT name FROM sqlite_master "
                    "WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name"
                ).fetchall()
            ]

            for table_name in database_tables:
                for row in connection.execute(f'PRAGMA table_info("{table_name}")').fetchall():
                    cid, name, data_type, not_null, default_value, primary_key = row
                    table_info_rows.append(
                        {
                            "table_name": table_name,
                            "column_order": cid + 1,
                            "column_name": name,
                            "data_type": data_type,
                            "not_null": "是" if not_null else "否",
                            "default_value": default_value,
                            "primary_key": "是" if primary_key else "否",
                            "business_description": COLUMN_DESCRIPTIONS.get(name, "待补充"),
                        }
                    )

            if table_exists(connection, "daily_bar"):
                source_status_df = pd.read_sql_query(
                    '''
                    SELECT source, COUNT(*) AS rows,
                           COUNT(DISTINCT symbol) AS symbols,
                           MIN(trade_date) AS first_date,
                           MAX(trade_date) AS last_date,
                           MAX(ingested_at) AS last_ingested_at
                    FROM daily_bar GROUP BY source ORDER BY source
                    ''',
                    connection,
                )
                choice_df = pd.read_sql_query(
                    "SELECT * FROM daily_bar WHERE source='choice' ORDER BY symbol, trade_date",
                    connection,
                )

            if table_exists(connection, "ingestion_run"):
                ingestion_df = pd.read_sql_query(
                    "SELECT * FROM ingestion_run ORDER BY run_id DESC LIMIT 100",
                    connection,
                )
    except sqlite3.DatabaseError as exc:
        db_integrity = f"读取失败：{type(exc).__name__}: {exc}"

table_info_df = pd.DataFrame(table_info_rows, columns=TABLE_INFO_COLUMNS)

choice_rows = len(choice_df)
choice_symbols = int(choice_df["symbol"].nunique()) if not choice_df.empty else 0
choice_first_date = str(choice_df["trade_date"].min()) if not choice_df.empty else ""
choice_last_date = str(choice_df["trade_date"].max()) if not choice_df.empty else ""

print("SQLite完整性：", db_integrity)
print("发现的数据表：", database_tables)
print("Choice真实落库行数：", choice_rows)
print("Choice证券数：", choice_symbols)
print("Choice日期范围：", choice_first_date or "无", "至", choice_last_date or "无")
display(source_status_df)
display(table_info_df.head(30))

SQLite完整性： ok
发现的数据表： ['daily_bar', 'ingestion_run']
Choice真实落库行数： 32
Choice证券数： 1
Choice日期范围： 2026-07-16 至 2026-08-28


,source,rows,symbols,first_date,last_date,last_ingested_at
0,choice,32,1,2026-07-16,2026-08-28,2026-09-01T01:48:04.632888+00:00
1,mock,42,2,2026-07-31,2026-08-28,2026-08-31T03:46:28.634076+00:00


,table_name,column_order,column_name,data_type,not_null,default_value,primary_key,business_description
0,daily_bar,1,source,TEXT,是,None,是,原始数据来源；Choice真实数据必须为choice
1,daily_bar,2,symbol,TEXT,是,None,是,证券代码，含交易所后缀
2,daily_bar,3,trade_date,TEXT,是,None,是,交易日期，ISO格式YYYY-MM-DD
3,daily_bar,4,adjustment,TEXT,是,'unadjusted',是,复权口径：unadjusted/qfq/hfq
4,daily_bar,5,open,REAL,否,None,否,开盘价
5,daily_bar,6,high,REAL,否,None,否,最高价
6,daily_bar,7,low,REAL,否,None,否,最低价
7,daily_bar,8,close,REAL,否,None,否,收盘价
8,daily_bar,9,volume,REAL,否,None,否,成交量，公司标准单位为股
9,daily_bar,10,amount,REAL,否,None,否,成交额，公司标准单位为CNY元


## 3. 检查 OpenBB Provider 注册和本地数据库读取

In [3]:
from importlib.metadata import PackageNotFoundError, version


def package_version(name: str) -> str:
    try:
        return version(name)
    except PackageNotFoundError:
        return "未安装"


openbb_imported = False
providers = set()
openbb_error = ""
openbb_read_status = "未执行"
openbb_read_rows = 0
openbb_read_symbol = ""

try:
    from openbb import obb

    openbb_imported = True
    providers = set(obb.coverage.providers)
except Exception as exc:
    openbb_error = f"{type(exc).__name__}: {exc}"

choice_registered = "choice" in providers
qianji_registered = "qianji" in providers

if openbb_imported and qianji_registered and not choice_df.empty:
    openbb_read_symbol = str(choice_df.iloc[0]["symbol"])
    symbol_rows = choice_df[choice_df["symbol"] == openbb_read_symbol]
    try:
        result = obb.equity.price.historical(
            symbol=openbb_read_symbol,
            start_date=str(symbol_rows["trade_date"].min()),
            end_date=str(symbol_rows["trade_date"].max()),
            source="choice",
            provider="qianji",
        )
        openbb_frame = result.to_dataframe().reset_index()
        openbb_read_rows = len(openbb_frame)
        openbb_read_status = "PASS" if openbb_read_rows == len(symbol_rows) else "PARTIAL"
    except Exception as exc:
        openbb_read_status = f"FAIL: {type(exc).__name__}: {exc}"
elif not openbb_imported:
    openbb_read_status = "SKIP: 当前环境无法导入OpenBB"
elif not qianji_registered:
    openbb_read_status = "SKIP: OpenBB未发现qianji Provider"
elif choice_df.empty:
    openbb_read_status = "SKIP: SQLite暂无Choice数据"

openbb_coverage_df = pd.DataFrame(
    [
        {
            "provider": "choice",
            "role": "直接调用Choice EmQuantAPI，不经过公司库",
            "expected_command": "equity.price.historical",
            "registered": "是" if choice_registered else "否",
            "package_version": package_version("openbb-choice"),
            "current_status": "02号Notebook已真实验证" if choice_registered else "需重新安装并执行openbb-build",
            "evidence": "provider=choice；日/周/月参数已实现，当前日线有真实验证证据",
        },
        {
            "provider": "qianji",
            "role": "从公司SQLite标准库读取，不重复消耗Choice流量",
            "expected_command": "equity.price.historical",
            "registered": "是" if qianji_registered else "否",
            "package_version": package_version("qianji-data-mini"),
            "current_status": openbb_read_status,
            "evidence": f"symbol={openbb_read_symbol or '-'}；返回{openbb_read_rows}行",
        },
    ]
)

print("OpenBB导入：", openbb_imported)
print("choice Provider已注册：", choice_registered)
print("qianji Provider已注册：", qianji_registered)
print("qianji本地读取：", openbb_read_status)
if openbb_error:
    print("OpenBB错误摘要：", openbb_error)
display(openbb_coverage_df)

OpenBB导入： True
choice Provider已注册： True
qianji Provider已注册： True
qianji本地读取： PASS


,provider,role,expected_command,registered,package_version,current_status,evidence
0,choice,直接调用Choice EmQuantAPI，不经过公司库,equity.price.historical,是,0.1.1,02号Notebook已真实验证,provider=choice；日/周/月参数已实现，当前日线有真实验证证据
1,qianji,从公司SQLite标准库读取，不重复消耗Choice流量,equity.price.historical,是,0.3.2,PASS,symbol=000001.SZ；返回32行


## 4. 生成 Choice 数据集地图

In [4]:
CHOICE_MANUAL_URL = "https://quantapi.eastmoney.com/Manual?from=web"
CHOICE_PYTHON_URL = "https://quantapi.eastmoney.com/Upload/EMQuantAPI_Python.html"
OPENBB_DOCS_URL = "https://docs.openbb.co/"

daily_status = "已真实验证" if choice_rows > 0 else "已实现待真实落库"
daily_evidence = (
    f"SQLite已有{choice_rows}行、{choice_symbols}只证券，"
    f"日期{choice_first_date}至{choice_last_date}"
    if choice_rows > 0
    else "当前扫描的SQLite未发现source=choice记录"
)

dataset_rows = [
    {
        "priority": "P0",
        "dataset_id": "CHOICE-P0-001",
        "dataset_name": "A股日线行情",
        "asset_scope": "沪深京股票；当前实证为000001.SZ",
        "choice_function": "csd",
        "choice_fields_or_report": "OPEN,HIGH,LOW,CLOSE,VOLUME,AMOUNT,PRECLOSE,PCTCHANGE",
        "frequency": "交易日收盘后增量",
        "incremental_anchor": "MAX(trade_date)+1",
        "raw_layer": "raw_json（当前）；raw_choice_csd_daily（后续独立原始层）",
        "standard_table": "daily_bar",
        "unique_key": "source+symbol+trade_date+adjustment",
        "openbb_model": "EquityHistorical",
        "openbb_route": "obb.equity.price.historical",
        "provider_path": "choice直连；qianji读库",
        "current_status": daily_status,
        "current_evidence": daily_evidence,
        "acceptance_evidence": "SQLite文件+Excel/JSON+qianji读取一致性",
        "permission_or_risk": "需持续确认日线权限、流量和复权口径",
        "official_source": CHOICE_PYTHON_URL,
    },
    {
        "priority": "P0",
        "dataset_id": "CHOICE-P0-002",
        "dataset_name": "A股周线/月线行情",
        "asset_scope": "沪深京股票",
        "choice_function": "csd（Period=2/3）",
        "choice_fields_or_report": "同日线字段",
        "frequency": "周末/月末增量",
        "incremental_anchor": "按周期末日期",
        "raw_layer": "待设计",
        "standard_table": "现有daily_bar不适合直接混存周期数据",
        "unique_key": "需增加period后再确定",
        "openbb_model": "EquityHistorical",
        "openbb_route": "obb.equity.price.historical(period=weekly/monthly)",
        "provider_path": "choice直连已写代码；qianji未落库",
        "current_status": "已实现未真实验证",
        "current_evidence": "代码支持Period映射，尚无本机真实结果证据",
        "acceptance_evidence": "固定股票日/周/月频率内容核对",
        "permission_or_risk": "必须避免把日线误当周线/月线；落库主键需扩展period",
        "official_source": CHOICE_PYTHON_URL,
    },
    {
        "priority": "P0",
        "dataset_id": "CHOICE-P0-003",
        "dataset_name": "证券主数据",
        "asset_scope": "股票、指数、基金等证券代码和基础属性",
        "choice_function": "ctr（StockInfo）/ css",
        "choice_fields_or_report": "字段需根据本机指标手册确定",
        "frequency": "每日增量+每周全量核对",
        "incremental_anchor": "上市状态/更新时间",
        "raw_layer": "raw_choice_security_master",
        "standard_table": "security_master",
        "unique_key": "source+symbol+effective_date",
        "openbb_model": "EquitySearch/公司自定义主数据模型（待核对）",
        "openbb_route": "待实现",
        "provider_path": "choice→标准库→qianji",
        "current_status": "规划中",
        "current_evidence": "官方手册确认ctr支持专题报表，示例含StockInfo",
        "acceptance_evidence": "全市场代码数、上市/退市状态、交易所抽样核对",
        "permission_or_risk": "具体报表字段依赖Choice指标手册与账号权限",
        "official_source": CHOICE_PYTHON_URL,
    },
    {
        "priority": "P0",
        "dataset_id": "CHOICE-P0-004",
        "dataset_name": "交易日历",
        "asset_scope": "沪深京及后续其他市场",
        "choice_function": "tradedates",
        "choice_fields_or_report": "日期序列；Market=CNSESH/CNSESZ等",
        "frequency": "每日检查；年度提前同步",
        "incremental_anchor": "MAX(calendar_date)+1",
        "raw_layer": "raw_choice_tradedates",
        "standard_table": "market_calendar",
        "unique_key": "source+market+calendar_date",
        "openbb_model": "公司自定义Calendar模型（待设计）",
        "openbb_route": "待实现",
        "provider_path": "choice→标准库→qianji",
        "current_status": "规划中",
        "current_evidence": "官方Python手册已确认tradedates接口及市场参数",
        "acceptance_evidence": "节假日、跨年、沪深市场日期抽样核对",
        "permission_or_risk": "不建议直接请求未来交易日；需明确市场代码",
        "official_source": CHOICE_PYTHON_URL,
    },
    {
        "priority": "P0",
        "dataset_id": "CHOICE-P0-005",
        "dataset_name": "三大财务报表",
        "asset_scope": "A股上市公司",
        "choice_function": "css/ctr",
        "choice_fields_or_report": "利润表、资产负债表、现金流字段待指标手册确认",
        "frequency": "公告日增量+季度回补",
        "incremental_anchor": "公告日期+报告期+版本",
        "raw_layer": "raw_choice_financials",
        "standard_table": "income_statement/balance_sheet/cash_flow",
        "unique_key": "source+symbol+period_ending+report_type+version",
        "openbb_model": "IncomeStatement/BalanceSheet/CashFlowStatement",
        "openbb_route": "待实现",
        "provider_path": "choice→标准库→qianji",
        "current_status": "规划中",
        "current_evidence": "官方手册确认css支持财务截面；具体指标未验证",
        "acceptance_evidence": "固定公司、报告期、合并口径和公告日期核对",
        "permission_or_risk": "单季/累计、合并/母公司、原始/调整后口径必须明确",
        "official_source": CHOICE_PYTHON_URL,
    },
    {
        "priority": "P1",
        "dataset_id": "CHOICE-P1-006",
        "dataset_name": "每日估值指标",
        "asset_scope": "A股及指数/ETF",
        "choice_function": "css/csd",
        "choice_fields_or_report": "PE、PB、PS、市值等字段待指标手册确认",
        "frequency": "交易日收盘后增量",
        "incremental_anchor": "trade_date",
        "raw_layer": "raw_choice_valuation",
        "standard_table": "equity_valuation_daily",
        "unique_key": "source+symbol+trade_date",
        "openbb_model": "待按OpenBB标准模型核对",
        "openbb_route": "待实现",
        "provider_path": "choice→标准库→qianji",
        "current_status": "规划中",
        "current_evidence": "官方手册确认css支持估值截面；指标权限未验证",
        "acceptance_evidence": "单位、TTM/静态/动态口径及空值核对",
        "permission_or_risk": "估值口径差异较大，字段名不能只看中文含义",
        "official_source": CHOICE_PYTHON_URL,
    },
    {
        "priority": "P1",
        "dataset_id": "CHOICE-P1-007",
        "dataset_name": "公司新闻与公告",
        "asset_scope": "股票及行业/板块",
        "choice_function": "cfn",
        "choice_fields_or_report": "companynews,industrynews,report,regularreport,tradeinfo",
        "frequency": "小时级增量或每日批量",
        "incremental_anchor": "datetime/eitime+infoCode",
        "raw_layer": "raw_choice_news",
        "standard_table": "company_news",
        "unique_key": "source+info_code",
        "openbb_model": "CompanyNews或公司自定义资讯模型（待核对）",
        "openbb_route": "待实现",
        "provider_path": "choice→标准库→qianji",
        "current_status": "规划中",
        "current_evidence": "官方手册已确认cfn内容类型和返回字段",
        "acceptance_evidence": "标题、时间、证券、来源、链接与资讯编码抽样核对",
        "permission_or_risk": "资讯流量、授权保存范围和去重规则需单独确认",
        "official_source": CHOICE_PYTHON_URL,
    },
    {
        "priority": "P1",
        "dataset_id": "CHOICE-P1-008",
        "dataset_name": "指数与ETF日线",
        "asset_scope": "指数、ETF",
        "choice_function": "csd",
        "choice_fields_or_report": "OPEN,HIGH,LOW,CLOSE,VOLUME,AMOUNT等",
        "frequency": "交易日收盘后增量",
        "incremental_anchor": "MAX(trade_date)+1",
        "raw_layer": "raw_choice_csd_daily",
        "standard_table": "index_daily/etf_daily（或扩展统一行情表）",
        "unique_key": "source+symbol+trade_date+adjustment",
        "openbb_model": "IndexHistorical/EquityHistorical（待按资产类型核对）",
        "openbb_route": "待验证后绑定",
        "provider_path": "可复用Choice csd适配逻辑",
        "current_status": "接口可复用，未真实验证",
        "current_evidence": "官方手册说明csd覆盖股票、指数、基金等品种",
        "acceptance_evidence": "指数、ETF各固定一个代码真实调用并核对单位",
        "permission_or_risk": "不同资产的成交量、金额和复权含义可能不同",
        "official_source": CHOICE_PYTHON_URL,
    },
]

data_map_df = pd.DataFrame(dataset_rows)
display(data_map_df)

,priority,dataset_id,dataset_name,asset_scope,choice_function,choice_fields_or_report,frequency,incremental_anchor,raw_layer,standard_table,unique_key,openbb_model,openbb_route,provider_path,current_status,current_evidence,acceptance_evidence,permission_or_risk,official_source
0,P0,CHOICE-P0-001,A股日线行情,沪深京股票；当前实证为000001.SZ,csd,"OPEN,HIGH,LOW,CLOSE,VOLUME,AMOUNT,PRECLOSE,PCT...",交易日收盘后增量,MAX(trade_date)+1,raw_json（当前）；raw_choice_csd_daily（后续独立原始层）,daily_bar,source+symbol+trade_date+adjustment,EquityHistorical,obb.equity.price.historical,choice直连；qianji读库,已真实验证,SQLite已有32行、1只证券，日期2026-07-16至2026-08-28,SQLite文件+Excel/JSON+qianji读取一致性,需持续确认日线权限、流量和复权口径,https://quantapi.eastmoney.com/Upload/EMQuantA...
1,P0,CHOICE-P0-002,A股周线/月线行情,沪深京股票,csd（Period=2/3）,同日线字段,周末/月末增量,按周期末日期,待设计,现有daily_bar不适合直接混存周期数据,需增加period后再确定,EquityHistorical,obb.equity.price.historical(period=weekly/mont...,choice直连已写代码；qianji未落库,已实现未真实验证,代码支持Period映射，尚无本机真实结果证据,固定股票日/周/月频率内容核对,必须避免把日线误当周线/月线；落库主键需扩展period,https://quantapi.eastmoney.com/Upload/EMQuantA...
2,P0,CHOICE-P0-003,证券主数据,股票、指数、基金等证券代码和基础属性,ctr（StockInfo）/ css,字段需根据本机指标手册确定,每日增量+每周全量核对,上市状态/更新时间,raw_choice_security_master,security_master,source+symbol+effective_date,EquitySearch/公司自定义主数据模型（待核对）,待实现,choice→标准库→qianji,规划中,官方手册确认ctr支持专题报表，示例含StockInfo,全市场代码数、上市/退市状态、交易所抽样核对,具体报表字段依赖Choice指标手册与账号权限,https://quantapi.eastmoney.com/Upload/EMQuantA...
3,P0,CHOICE-P0-004,交易日历,沪深京及后续其他市场,tradedates,日期序列；Market=CNSESH/CNSESZ等,每日检查；年度提前同步,MAX(calendar_date)+1,raw_choice_tradedates,market_calendar,source+market+calendar_date,公司自定义Calendar模型（待设计）,待实现,choice→标准库→qianji,规划中,官方Python手册已确认tradedates接口及市场参数,节假日、跨年、沪深市场日期抽样核对,不建议直接请求未来交易日；需明确市场代码,https://quantapi.eastmoney.com/Upload/EMQuantA...
4,P0,CHOICE-P0-005,三大财务报表,A股上市公司,css/ctr,利润表、资产负债表、现金流字段待指标手册确认,公告日增量+季度回补,公告日期+报告期+版本,raw_choice_financials,income_statement/balance_sheet/cash_flow,source+symbol+period_ending+report_type+version,IncomeStatement/BalanceSheet/CashFlowStatement,待实现,choice→标准库→qianji,规划中,官方手册确认css支持财务截面；具体指标未验证,固定公司、报告期、合并口径和公告日期核对,单季/累计、合并/母公司、原始/调整后口径必须明确,https://quantapi.eastmoney.com/Upload/EMQuantA...
5,P1,CHOICE-P1-006,每日估值指标,A股及指数/ETF,css/csd,PE、PB、PS、市值等字段待指标手册确认,交易日收盘后增量,trade_date,raw_choice_valuation,equity_valuation_daily,source+symbol+trade_date,待按OpenBB标准模型核对,待实现,choice→标准库→qianji,规划中,官方手册确认css支持估值截面；指标权限未验证,单位、TTM/静态/动态口径及空值核对,估值口径差异较大，字段名不能只看中文含义,https://quantapi.eastmoney.com/Upload/EMQuantA...
6,P1,CHOICE-P1-007,公司新闻与公告,股票及行业/板块,cfn,"companynews,industrynews,report,regularreport,...",小时级增量或每日批量,datetime/eitime+infoCode,raw_choice_news,company_news,source+info_code,CompanyNews或公司自定义资讯模型（待核对）,待实现,choice→标准库→qianji,规划中,官方手册已确认cfn内容类型和返回字段,标题、时间、证券、来源、链接与资讯编码抽样核对,资讯流量、授权保存范围和去重规则需单独确认,https://quantapi.eastmoney.com/Upload/EMQuantA...
7,P1,CHOICE-P1-008,指数与ETF日线,指数、ETF,csd,"OPEN,HIGH,LOW,CLOSE,VOLUME,AMOUNT等",交易日收盘后增量,MAX(trade_date)+1,raw_choice_csd_daily,index_daily/etf_daily（或扩展统一行情表）,source+symbol+trade_date+adjustment,IndexHistorical/EquityHistorical（待按资产类型核对）,待验证后绑定,可复用Choice csd适配逻辑,接口可复用，未真实验证,官方手册说明csd覆盖股票、指数、基金等品种,指数、ETF各固定一个代码真实调用并核对单位,不同资产的成交量、金额和复权含义可能不同,https://quantapi.eastmoney.com/Upload/EMQuantA...


## 5. 生成已实现日线的字段映射表

In [5]:
volume_multiplier = float(os.getenv("CHOICE_VOLUME_MULTIPLIER", "1"))
amount_multiplier = float(os.getenv("CHOICE_AMOUNT_MULTIPLIER", "1"))

field_mapping_rows = [
    ["请求参数", "symbol", "TEXT", "symbol", "symbol", "转大写并保留交易所后缀", "证券代码", "否", "000001.SZ格式"],
    ["Dates", "trade_date", "TEXT", "date", "date", "支持YYYY/M/D等输入，统一为YYYY-MM-DD", "日期", "否", "可解析且在请求区间内"],
    ["OPEN", "open", "REAL", "open", "open", "转浮点数", "CNY/股", "是", "若非空则>=0"],
    ["HIGH", "high", "REAL", "high", "high", "转浮点数", "CNY/股", "是", "不低于open/low/close"],
    ["LOW", "low", "REAL", "low", "low", "转浮点数", "CNY/股", "是", "不高于open/high/close"],
    ["CLOSE", "close", "REAL", "close", "close", "转浮点数", "CNY/股", "是", "若非空则>=0"],
    ["VOLUME", "volume", "REAL", "volume", "volume", f"乘以CHOICE_VOLUME_MULTIPLIER={volume_multiplier:g}", "share", "是", "非负；与终端抽样核对"],
    ["AMOUNT", "amount", "REAL", "amount", "amount", f"乘以CHOICE_AMOUNT_MULTIPLIER={amount_multiplier:g}", "CNY", "是", "非负；量价数量级合理"],
    ["PRECLOSE", "previous_close", "REAL", "prev_close", "previous_close", "转浮点数", "CNY/股", "是", "与前一交易日收盘按复权口径核对"],
    ["PCTCHANGE", "change_percent", "REAL", "change_percent", "change_percent", "SQLite保留百分点；OpenBB输出时除以100", "百分点→小数", "是", "1.25%在OpenBB中为0.0125"],
    ["派生", "change", "不落库", "change", "不直接提供", "close-previous_close", "CNY/股", "是", "与价格差一致"],
    ["固定值", "source", "TEXT", "source", "source", "固定为choice", "标识", "否", "不得写成manual/mock"],
    ["固定值", "currency", "TEXT", "currency", "currency", "A股默认CNY", "币种", "否", "CNY"],
    ["固定值", "timezone", "TEXT", "timezone", "timezone", "A股默认Asia/Shanghai", "IANA时区", "否", "Asia/Shanghai"],
    ["固定值", "volume_unit", "TEXT", "volume_unit", "volume_unit", "标准层固定share", "单位", "否", "share"],
    ["固定值", "amount_unit", "TEXT", "amount_unit", "amount_unit", "标准层固定CNY", "单位", "否", "CNY"],
    ["请求参数", "adjustment", "TEXT", "adjustment", "不直接返回", "None→unadjusted，qfq/hfq保持", "口径", "否", "与Choice AdjustFlag一致"],
    ["系统生成", "ingested_at", "TEXT", "不返回", "ingested_at", "写入时生成UTC ISO时间", "时间戳", "否", "可解析且有时区"],
    ["原始返回", "raw_json", "TEXT", "不返回", "raw", "JSON原样保留用于追溯，不含账号密码", "JSON", "是", "可回溯供应商字段"],
]

field_mapping_df = pd.DataFrame(
    field_mapping_rows,
    columns=[
        "choice_source", "standard_field", "sqlite_type", "choice_provider_field",
        "qianji_provider_field", "conversion_rule", "standard_unit", "nullable",
        "acceptance_rule",
    ],
)
display(field_mapping_df)

,choice_source,standard_field,sqlite_type,choice_provider_field,qianji_provider_field,conversion_rule,standard_unit,nullable,acceptance_rule
0,请求参数,symbol,TEXT,symbol,symbol,转大写并保留交易所后缀,证券代码,否,000001.SZ格式
1,Dates,trade_date,TEXT,date,date,支持YYYY/M/D等输入，统一为YYYY-MM-DD,日期,否,可解析且在请求区间内
2,OPEN,open,REAL,open,open,转浮点数,CNY/股,是,若非空则>=0
3,HIGH,high,REAL,high,high,转浮点数,CNY/股,是,不低于open/low/close
4,LOW,low,REAL,low,low,转浮点数,CNY/股,是,不高于open/high/close
5,CLOSE,close,REAL,close,close,转浮点数,CNY/股,是,若非空则>=0
6,VOLUME,volume,REAL,volume,volume,乘以CHOICE_VOLUME_MULTIPLIER=1,share,是,非负；与终端抽样核对
7,AMOUNT,amount,REAL,amount,amount,乘以CHOICE_AMOUNT_MULTIPLIER=1,CNY,是,非负；量价数量级合理
8,PRECLOSE,previous_close,REAL,prev_close,previous_close,转浮点数,CNY/股,是,与前一交易日收盘按复权口径核对
9,PCTCHANGE,change_percent,REAL,change_percent,change_percent,SQLite保留百分点；OpenBB输出时除以100,百分点→小数,是,1.25%在OpenBB中为0.0125


## 6. 自动计算当前数据质量门槛

In [6]:
def quality_row(name, hard_threshold, actual, passed, evidence_source):
    if passed is None:
        status = "SKIP"
    else:
        status = "PASS" if passed else "FAIL"
    return {
        "check": name,
        "hard_threshold": hard_threshold,
        "actual": actual,
        "status": status,
        "evidence_source": evidence_source,
    }


quality_rows = []
has_choice = not choice_df.empty

if has_choice:
    key_fields = ["source", "symbol", "trade_date", "adjustment"]
    required_fields = ["source", "symbol", "trade_date", "open", "high", "low", "close"]
    duplicate_count = int(choice_df.duplicated(subset=key_fields).sum())
    missing_count = int(choice_df[required_fields].isna().any(axis=1).sum())

    ohlc = choice_df[["open", "high", "low", "close"]].apply(pd.to_numeric, errors="coerce")
    complete = ohlc.notna().all(axis=1)
    ohlc_errors = int(
        (
            complete
            & (
                (ohlc["high"] < ohlc.max(axis=1))
                | (ohlc["low"] > ohlc.min(axis=1))
            )
        ).sum()
    )
    negative_volume = int((pd.to_numeric(choice_df["volume"], errors="coerce").dropna() < 0).sum())
    unit_ok = (
        set(choice_df["volume_unit"].dropna().astype(str)) <= {"share"}
        and set(choice_df["amount_unit"].dropna().astype(str)) <= {"CNY"}
    )
else:
    duplicate_count = missing_count = ohlc_errors = negative_volume = None
    unit_ok = None

quality_rows.extend(
    [
        quality_row("Choice真实数据非空", ">0行", choice_rows, choice_rows > 0, "daily_bar/source=choice"),
        quality_row("标准主键唯一", "重复数=0", duplicate_count if has_choice else "未检查", duplicate_count == 0 if has_choice else None, "daily_bar"),
        quality_row("关键字段完整", "缺失行数=0", missing_count if has_choice else "未检查", missing_count == 0 if has_choice else None, "daily_bar"),
        quality_row("OHLC逻辑正确", "异常数=0", ohlc_errors if has_choice else "未检查", ohlc_errors == 0 if has_choice else None, "daily_bar"),
        quality_row("成交量非负", "负数=0", negative_volume if has_choice else "未检查", negative_volume == 0 if has_choice else None, "daily_bar"),
        quality_row("标准单位统一", "volume=share；amount=CNY", "符合" if unit_ok else ("不符合" if unit_ok is False else "未检查"), unit_ok, "daily_bar"),
        quality_row("SQLite完整性", "PRAGMA quick_check=ok", db_integrity, db_integrity == "ok", str(DB_PATH)),
        quality_row("Choice Provider注册", "choice在obb.coverage.providers", choice_registered, choice_registered, "OpenBB coverage"),
        quality_row("qianji Provider注册", "qianji在obb.coverage.providers", qianji_registered, qianji_registered, "OpenBB coverage"),
        quality_row("qianji读取Choice落库数据", "读取行数=同证券数据库行数", openbb_read_status, openbb_read_status == "PASS", "provider=qianji, source=choice"),
    ]
)

quality_df = pd.DataFrame(quality_rows)
display(quality_df)

,check,hard_threshold,actual,status,evidence_source
0,Choice真实数据非空,>0行,32,PASS,daily_bar/source=choice
1,标准主键唯一,重复数=0,0,PASS,daily_bar
2,关键字段完整,缺失行数=0,0,PASS,daily_bar
3,OHLC逻辑正确,异常数=0,0,PASS,daily_bar
4,成交量非负,负数=0,0,PASS,daily_bar
5,标准单位统一,volume=share；amount=CNY,符合,PASS,daily_bar
6,SQLite完整性,PRAGMA quick_check=ok,ok,PASS,D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\data\...
7,Choice Provider注册,choice在obb.coverage.providers,True,PASS,OpenBB coverage
8,qianji Provider注册,qianji在obb.coverage.providers,True,PASS,OpenBB coverage
9,qianji读取Choice落库数据,读取行数=同证券数据库行数,PASS,PASS,"provider=qianji, source=choice"


## 7. 生成状态说明和可执行的下一步

In [7]:
status_legend_df = pd.DataFrame(
    [
        ["已真实验证", "代码已运行，真实供应商数据已落库，并有可复核证据", "可作为当前开工硬证据；仍不等于生产全量完成"],
        ["已实现未真实验证", "已有代码，但没有当前电脑/账号的真实结果证据", "需固定小样本调用、落库和核对"],
        ["接口可复用，未真实验证", "现有通用逻辑理论上覆盖该资产，但尚未抽样", "至少选择一个固定代码真实验证"],
        ["规划中", "已明确数据集、接口方向、目标表和验收方式，尚未开发", "按P0/P1顺序进入开发"],
        ["PASS", "自动质量门槛通过", "可将本次输出作为证据附件"],
        ["FAIL", "自动质量门槛未通过", "先保存错误摘要，再修复或补证据"],
        ["SKIP", "当前环境缺数据或组件，未执行检查", "不能当作通过"],
    ],
    columns=["status", "definition", "management_meaning"],
)

next_steps_df = pd.DataFrame(
    [
        [1, "补齐Choice日线数据地图证据", "本Notebook导出的Excel/JSON", "今天", "当前执行人", "已完成/进行中"],
        [2, "扩展日线验证集", "000001.SZ、601988.SH、510300.SH各近30交易日", "下一工作日", "当前执行人", "待执行"],
        [3, "确认证券主数据字段", "Choice指标手册字段清单+5条样本", "下一工作日", "当前执行人/组长确认口径", "待执行"],
        [4, "开发交易日历最小插件", "tradedates真实返回+market_calendar落库", "P0下一项", "当前执行人", "待执行"],
        [5, "确认公司正式数据库目标", "连接方式、库名、schema、表命名和写权限", "转生产前", "组长/数据库负责人", "待协助"],
    ],
    columns=["order", "task", "acceptance_evidence", "target_time", "owner", "status"],
)

usage_df = pd.DataFrame(
    [
        ["生成时间", pd.Timestamp.now(tz="UTC").isoformat()],
        ["项目目录", str(PROJECT_ROOT)],
        ["SQLite数据库", str(DB_PATH)],
        ["数据库完整性", db_integrity],
        ["Choice落库行数", choice_rows],
        ["Choice证券数", choice_symbols],
        ["Choice日期范围", f"{choice_first_date or '-'} 至 {choice_last_date or '-'}"],
        ["凭据是否导出", "否"],
        ["地图边界", "当前为Choice插件与轻量SQLite MVP，不等同于公司生产全库"],
        ["Choice官方手册", CHOICE_MANUAL_URL],
        ["Choice Python手册", CHOICE_PYTHON_URL],
        ["OpenBB开发文档", OPENBB_DOCS_URL],
    ],
    columns=["item", "value"],
)

display(next_steps_df)
display(status_legend_df)

,order,task,acceptance_evidence,target_time,owner,status
0,1,补齐Choice日线数据地图证据,本Notebook导出的Excel/JSON,今天,当前执行人,已完成/进行中
1,2,扩展日线验证集,000001.SZ、601988.SH、510300.SH各近30交易日,下一工作日,当前执行人,待执行
2,3,确认证券主数据字段,Choice指标手册字段清单+5条样本,下一工作日,当前执行人/组长确认口径,待执行
3,4,开发交易日历最小插件,tradedates真实返回+market_calendar落库,P0下一项,当前执行人,待执行
4,5,确认公司正式数据库目标,连接方式、库名、schema、表命名和写权限,转生产前,组长/数据库负责人,待协助


,status,definition,management_meaning
0,已真实验证,代码已运行，真实供应商数据已落库，并有可复核证据,可作为当前开工硬证据；仍不等于生产全量完成
1,已实现未真实验证,已有代码，但没有当前电脑/账号的真实结果证据,需固定小样本调用、落库和核对
2,接口可复用，未真实验证,现有通用逻辑理论上覆盖该资产，但尚未抽样,至少选择一个固定代码真实验证
3,规划中,已明确数据集、接口方向、目标表和验收方式，尚未开发,按P0/P1顺序进入开发
4,PASS,自动质量门槛通过,可将本次输出作为证据附件
5,FAIL,自动质量门槛未通过,先保存错误摘要，再修复或补证据
6,SKIP,当前环境缺数据或组件，未执行检查,不能当作通过


## 8. 导出 Excel 和 JSON 数据地图

In [8]:
import json
from datetime import datetime

from openpyxl import load_workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter


timestamp = datetime.now().astimezone().strftime("%Y%m%d_%H%M%S")
excel_path = OUTPUT_DIR / f"Choice数据地图_{timestamp}.xlsx"
json_path = OUTPUT_DIR / f"Choice数据地图_{timestamp}.json"

sheet_frames = {
    "使用说明": usage_df,
    "数据集地图": data_map_df,
    "字段映射": field_mapping_df,
    "库表字段": table_info_df,
    "源落库统计": source_status_df,
    "最近采集记录": ingestion_df,
    "质量门槛": quality_df,
    "OpenBB覆盖": openbb_coverage_df,
    "下一步": next_steps_df,
    "状态说明": status_legend_df,
}

sheet_subtitles = {
    "使用说明": "本次自动扫描的范围、数据位置和官方资料来源",
    "数据集地图": "Choice数据集、接口、库表、OpenBB模型、管理状态和验收证据",
    "字段映射": "已实现A股日线从Choice原始字段到SQLite/OpenBB标准字段的转换",
    "库表字段": "从当前SQLite数据库PRAGMA自动读取的真实表结构",
    "源落库统计": "各source的行数、证券数、日期范围和最近入库时间",
    "最近采集记录": "最近100次采集任务；错误摘要不应包含任何凭据",
    "质量门槛": "硬指标自动检查；SKIP不能视为PASS",
    "OpenBB覆盖": "choice直连Provider与qianji读库Provider的注册/读取状态",
    "下一步": "由当前数据地图直接生成的P0推进清单",
    "状态说明": "明确区分事实、已实现能力和规划，防止进度表述失真",
}


def safe_value(value):
    if isinstance(value, str) and value[:1] in {"=", "+", "-", "@"}:
        return "'" + value
    return value


safe_frames = {}
for sheet_name, frame in sheet_frames.items():
    safe_frame = frame.copy()
    for column in safe_frame.columns:
        safe_frame[column] = safe_frame[column].map(safe_value)
    safe_frames[sheet_name] = safe_frame

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    for sheet_name, frame in safe_frames.items():
        frame.to_excel(writer, sheet_name=sheet_name, index=False, startrow=3)

workbook = load_workbook(excel_path)

NAVY = "17365D"
BLUE = "2F75B5"
LIGHT_BLUE = "D9EAF7"
GREEN = "E2F0D9"
YELLOW = "FFF2CC"
RED = "FCE4D6"
GRAY = "E7E6E6"
WHITE = "FFFFFF"
GRID = "B7C9D6"
thin = Side(style="thin", color=GRID)

for worksheet in workbook.worksheets:
    frame = safe_frames[worksheet.title]
    max_col = max(1, len(frame.columns))
    max_row = worksheet.max_row
    last_col = get_column_letter(max_col)

    worksheet.merge_cells(start_row=1, start_column=1, end_row=1, end_column=max_col)
    title_cell = worksheet.cell(1, 1, f"Choice数据地图｜{worksheet.title}")
    title_cell.fill = PatternFill("solid", fgColor=NAVY)
    title_cell.font = Font(name="Microsoft YaHei", size=15, bold=True, color=WHITE)
    title_cell.alignment = Alignment(horizontal="left", vertical="center")
    worksheet.row_dimensions[1].height = 28

    worksheet.merge_cells(start_row=2, start_column=1, end_row=2, end_column=max_col)
    subtitle_cell = worksheet.cell(2, 1, sheet_subtitles[worksheet.title])
    subtitle_cell.fill = PatternFill("solid", fgColor=LIGHT_BLUE)
    subtitle_cell.font = Font(name="Microsoft YaHei", size=10, color=NAVY)
    subtitle_cell.alignment = Alignment(wrap_text=True, vertical="center")
    worksheet.row_dimensions[2].height = 30

    header_row = 4
    for cell in worksheet[header_row]:
        cell.fill = PatternFill("solid", fgColor=BLUE)
        cell.font = Font(name="Microsoft YaHei", bold=True, color=WHITE)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border = Border(top=thin, bottom=thin, left=thin, right=thin)
    worksheet.row_dimensions[header_row].height = 34

    for row in worksheet.iter_rows(min_row=header_row + 1, max_row=max_row, max_col=max_col):
        for cell in row:
            cell.font = Font(name="Microsoft YaHei", size=10)
            cell.alignment = Alignment(vertical="top", wrap_text=True)
            cell.border = Border(top=thin, bottom=thin, left=thin, right=thin)
            text = str(cell.value or "")
            if text in {"PASS", "已真实验证"} or "已真实验证" in text:
                cell.fill = PatternFill("solid", fgColor=GREEN)
            elif text.startswith("FAIL"):
                cell.fill = PatternFill("solid", fgColor=RED)
            elif text in {"SKIP", "规划中", "待执行", "待协助"} or "未真实验证" in text:
                cell.fill = PatternFill("solid", fgColor=YELLOW)
            elif text == "已完成/进行中":
                cell.fill = PatternFill("solid", fgColor=LIGHT_BLUE)

            if text.startswith("https://"):
                cell.hyperlink = text
                cell.style = "Hyperlink"

    for column_index, column_name in enumerate(frame.columns, start=1):
        values = [str(column_name)] + [str(value or "") for value in frame[column_name].head(200)]
        longest = max((len(value) for value in values), default=8)
        width = min(max(longest * 1.15 + 2, 10), 42)
        worksheet.column_dimensions[get_column_letter(column_index)].width = width

    worksheet.freeze_panes = "A5"
    worksheet.auto_filter.ref = f"A4:{last_col}{max_row}"
    worksheet.sheet_view.showGridLines = False
    worksheet.auto_filter.ref = f"A4:{last_col}{max_row}"
    worksheet.print_title_rows = "1:4"
    worksheet.page_setup.orientation = "landscape"
    worksheet.page_setup.fitToWidth = 1
    worksheet.sheet_properties.pageSetUpPr.fitToPage = True

workbook.save(excel_path)

json_payload = {
    "generated_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "project_root": str(PROJECT_ROOT),
    "database": str(DB_PATH),
    "credentials_included": False,
    "database_integrity": db_integrity,
    "choice_rows": choice_rows,
    "choice_symbols": choice_symbols,
    "datasets": data_map_df.to_dict(orient="records"),
    "field_mapping": field_mapping_df.to_dict(orient="records"),
    "database_schema": table_info_df.where(pd.notna(table_info_df), None).to_dict(orient="records"),
    "source_status": source_status_df.where(pd.notna(source_status_df), None).to_dict(orient="records"),
    "quality_gates": quality_df.where(pd.notna(quality_df), None).to_dict(orient="records"),
    "openbb_coverage": openbb_coverage_df.to_dict(orient="records"),
}
json_path.write_text(json.dumps(json_payload, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

print("Excel已生成：", excel_path)
print("JSON已生成：", json_path)
print("Excel大小（字节）：", excel_path.stat().st_size)
print("JSON大小（字节）：", json_path.stat().st_size)

Excel已生成： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\validation_output\Choice数据地图_20260831_191404.xlsx
JSON已生成： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\validation_output\Choice数据地图_20260831_191404.json
Excel大小（字节）： 25604
JSON大小（字节）： 27762


## 9. 自动复核导出文件

In [9]:
from openpyxl import load_workbook

expected_sheets = list(sheet_frames)
check_workbook = load_workbook(excel_path, read_only=True, data_only=False)
actual_sheets = check_workbook.sheetnames
missing_sheets = sorted(set(expected_sheets) - set(actual_sheets))

json_check = json.loads(json_path.read_text(encoding="utf-8"))
checks = pd.DataFrame(
    [
        ["Excel文件存在", excel_path.exists(), str(excel_path)],
        ["JSON文件存在", json_path.exists(), str(json_path)],
        ["工作表完整", not missing_sheets, f"缺少：{missing_sheets}" if missing_sheets else f"共{len(actual_sheets)}张表"],
        ["数据集地图非空", len(data_map_df) > 0, f"{len(data_map_df)}个数据集"],
        ["字段映射非空", len(field_mapping_df) > 0, f"{len(field_mapping_df)}个字段/元数据项"],
        ["凭据未导出", json_check.get("credentials_included") is False, "credentials_included=False"],
    ],
    columns=["check", "passed", "detail"],
)

display(checks)
if not checks["passed"].all():
    raise RuntimeError("导出复核未全部通过，请查看上表。")

print("数据地图生成与复核完成。你可以把Excel和JSON作为会议/日报附件。")

,check,passed,detail
0,Excel文件存在,True,D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\valid...
1,JSON文件存在,True,D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\valid...
2,工作表完整,True,共10张表
3,数据集地图非空,True,8个数据集
4,字段映射非空,True,19个字段/元数据项
5,凭据未导出,True,credentials_included=False


数据地图生成与复核完成。你可以把Excel和JSON作为会议/日报附件。
